# Chapter 4: Optimizing with torch.compile and Kernel Fusion
### Companion notebook (High-Performance Machine Learning)

This notebook makes the chapter's claims measurable on a real GPU. It is written for a
**Colab T4** (Runtime → Change runtime type → T4 GPU), the same card used in the
running RAG case study.

What you will measure here:

1. **Graph break diagnosis**: use `torch._dynamo.explain` to find and fix breaks in a real model.
2. **Profiler comparison**: profile eager vs compiled GPT-2 and see fusion in the kernel table.
3. **Custom operator registration**: register a custom op with `torch.library` so it compiles cleanly.
4. **Warm-up timing**: measure cold compile vs warm execution and the effect of regional compilation.

> Solutions and extended versions of the exercises live at
> https://github.com/PacktPublishing/High-Performance-Machine-Learning

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import time
import statistics
import numpy as np
from torch.profiler import profile, ProfilerActivity

assert torch.cuda.is_available(), 'Enable a GPU runtime (Colab: Runtime > Change runtime type > T4).'
dev = torch.device('cuda')
name = torch.cuda.get_device_name(0)
print(f'GPU: {name}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
input_ids = tokenizer('The key insight behind kernel fusion is that', return_tensors='pt').input_ids.to(dev)
print(f'Input shape: {input_ids.shape}')

## 1. Graph break diagnosis with `torch._dynamo.explain`

We wrap GPT-2 in a module that introduces two common graph breaks:
a `.item()` call (scalar extraction) and a `print()` (Python side effect).
Then we use `torch._dynamo.explain` to surface them.

In [ ]:
base_model = GPT2LMHeadModel.from_pretrained('gpt2').to(dev).eval()

class GPT2WithBreaks(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        embeds = self.model.transformer.wte(input_ids)
        # Graph break 1: extracts a runtime tensor value into Python
        max_val = embeds.abs().max().item()
        # Graph break 2: Python side effect
        print(f'max embedding magnitude: {max_val:.4f}')
        logits = self.model(inputs_embeds=embeds).logits
        return logits

broken_model = GPT2WithBreaks(base_model).to(dev).eval()

torch._dynamo.reset()
explanation = torch._dynamo.explain(broken_model)(input_ids)
print('\n--- explain() output (broken model) ---')
print(f'Graph Count: {explanation.graph_count}')
print(f'Graph Break Count: {explanation.graph_break_count}')
print(f'Break Reasons:')
for i, reason in enumerate(explanation.break_reasons):
    print(f'  {i+1}. {reason.reason} at {reason.user_stack[-1]}')

In [ ]:
# Fix: remove the .item() and print()
class GPT2Clean(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        return self.model(input_ids).logits

clean_model = GPT2Clean(base_model).to(dev).eval()

torch._dynamo.reset()
explanation_clean = torch._dynamo.explain(clean_model)(input_ids)
print('--- explain() output (clean model) ---')
print(f'Graph Count: {explanation_clean.graph_count}')
print(f'Graph Break Count: {explanation_clean.graph_break_count}')
print('\nOne graph, zero breaks. The full forward compiles as a single region.')

## 2. Profiler comparison: eager vs compiled

We profile one forward pass in eager mode and one after `torch.compile`,
then compare kernel count and total CUDA time.

In [ ]:
# Eager profiling
torch._dynamo.reset()
eager_model = GPT2LMHeadModel.from_pretrained('gpt2').to(dev).eval()

with torch.no_grad():
    for _ in range(5):
        eager_model(input_ids)
    torch.cuda.synchronize()

with torch.no_grad():
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof_eager:
        eager_model(input_ids)
        torch.cuda.synchronize()

print('--- Eager mode (top 15 by CUDA time) ---')
print(prof_eager.key_averages().table(sort_by='cuda_time_total', row_limit=15))

eager_cuda_events = [e for e in prof_eager.key_averages() if e.device_time_total > 0]
total_eager = sum(e.device_time_total for e in prof_eager.key_averages())
print(f'\nDistinct kernel types: {len(eager_cuda_events)}')
print(f'Total CUDA time: {total_eager / 1000:.3f} ms')

In [ ]:
# Compiled profiling
torch._dynamo.reset()
compiled_model = torch.compile(GPT2LMHeadModel.from_pretrained('gpt2').to(dev).eval())

with torch.no_grad():
    for _ in range(5):
        compiled_model(input_ids)
    torch.cuda.synchronize()

with torch.no_grad():
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof_compiled:
        compiled_model(input_ids)
        torch.cuda.synchronize()

print('--- Compiled mode (top 15 by CUDA time) ---')
print(prof_compiled.key_averages().table(sort_by='cuda_time_total', row_limit=15))

compiled_cuda_events = [e for e in prof_compiled.key_averages() if e.device_time_total > 0]
total_compiled = sum(e.device_time_total for e in prof_compiled.key_averages())
print(f'\nDistinct kernel types: {len(compiled_cuda_events)}')
print(f'Total CUDA time: {total_compiled / 1000:.3f} ms')
if total_compiled > 0:
    print(f'Speedup: {total_eager / total_compiled:.2f}x')
else:
    print('Speedup: profiler reported no compiled CUDA time on this run; '
          'time a warm forward directly to compare.')

## 3. Custom operator registration with `torch.library`

When you have a custom CUDA kernel that Dynamo cannot trace, you register it
with `torch.library` so it becomes a first-class graph node.

In [ ]:
# Define a custom library and operator
lib = torch.library.Library('mylib', 'DEF')
lib.define('scaled_sigmoid(Tensor x, float scale) -> Tensor')

# CUDA implementation (in practice, this would call your custom kernel)
@torch.library.impl(lib, 'scaled_sigmoid', 'CUDA')
def scaled_sigmoid_cuda(x, scale):
    return torch.sigmoid(x) * scale

# Meta implementation (FakeTensor support for the compiler)
@torch.library.impl(lib, 'scaled_sigmoid', 'Meta')
def scaled_sigmoid_meta(x, scale):
    return torch.empty_like(x)

# Use it in a model
W = torch.randn(64, 64, device=dev)

class ModelWithRegisteredOp(nn.Module):
    def forward(self, x):
        x = x @ W
        x = torch.ops.mylib.scaled_sigmoid(x, 2.0)
        x = x @ W.T
        return x

model_custom = ModelWithRegisteredOp().to(dev).eval()
torch._dynamo.reset()
explanation_custom = torch._dynamo.explain(model_custom)(torch.randn(1, 64, device=dev))
print(f'Graph Count: {explanation_custom.graph_count}')
print(f'Graph Break Count: {explanation_custom.graph_break_count}')
print('\nThe registered op compiles as a single graph, no breaks.')

## 4. Warm-up timing: cold compile vs warm execution

Measure the first-call compilation cost and how regional compilation reduces it.

In [ ]:
import torch._inductor.config as inductor_config
inductor_config.fx_graph_cache = False  # force cold compile

torch._dynamo.reset()
fresh_model = GPT2LMHeadModel.from_pretrained('gpt2').to(dev).eval()
compiled_fresh = torch.compile(fresh_model)

# Cold compile
torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    compiled_fresh(input_ids)
torch.cuda.synchronize()
cold_time = time.perf_counter() - t0

# Warm calls
warm_times = []
with torch.no_grad():
    for _ in range(20):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        compiled_fresh(input_ids)
        torch.cuda.synchronize()
        warm_times.append(time.perf_counter() - t0)

median_warm = statistics.median(warm_times)
print(f'First call (cold compile): {cold_time*1000:.0f} ms')
print(f'Subsequent calls (median): {median_warm*1000:.2f} ms')
print(f'Ratio: {cold_time/median_warm:.0f}x')

In [ ]:
# Regional compilation: compile per-layer
torch._dynamo.reset()
regional_model = GPT2LMHeadModel.from_pretrained('gpt2').to(dev).eval()

torch.cuda.synchronize()
t0 = time.perf_counter()
for i, layer in enumerate(regional_model.transformer.h):
    regional_model.transformer.h[i] = torch.compile(layer)
with torch.no_grad():
    regional_model(input_ids)
torch.cuda.synchronize()
regional_cold = time.perf_counter() - t0

# Warm calls
regional_warm = []
with torch.no_grad():
    for _ in range(20):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        regional_model(input_ids)
        torch.cuda.synchronize()
        regional_warm.append(time.perf_counter() - t0)

print(f'Regional cold compile (12 layers): {regional_cold*1000:.0f} ms')
print(f'Regional warm (median): {statistics.median(regional_warm)*1000:.2f} ms')
print(f'Cold-start reduction vs whole-model: {cold_time/regional_cold:.1f}x faster')
print(f'\nRegional compilation cuts cold-start by ~{cold_time/regional_cold:.0f}x because the code')
print('generator runs once and the cached result serves all 12 identical layers.')

## 5. Exercises

**4.1** Add a `torch.where` branch to the model's forward pass (e.g., `x = torch.where(x > 0, x, x * 0.01)`).
Run `torch._dynamo.explain` and confirm it does NOT cause a graph break (why not?).

**4.2** Compile GPT-2 with `mode="reduce-overhead"` and re-run the profiler comparison from Section 2.
Does the speedup increase? Time 20 consecutive forward passes and report the median.

**4.3** Mark the sequence dimension dynamic with `torch._dynamo.mark_dynamic(input_ids, 1)`
and compile the model. Feed it 5 different sequence lengths (32, 64, 128, 256, 512) and time each.
Compare against the same 5 lengths WITHOUT `mark_dynamic`. How many recompilations occur?

**4.4** In the warm-up section, enable the Inductor cache (`inductor_config.fx_graph_cache = True`
and set `TORCHINDUCTOR_CACHE_DIR`). Restart the runtime and measure the first-call time with a warm cache.
How much faster is it than the cold compile?

**4.5** (Stretch) Compile GPT-2 with `fullgraph=True`. If it raises, identify the graph break,
trace it to the source in the HuggingFace model code, and propose a fix or workaround.